In [1]:
import pandas as pd

ports = pd.DataFrame({
    "embark_town": ["Southampton", "Cherbourg", "Queenstown"],
    "pays": ["Angleterre", "France", "Irlande"]
})

ports

,embark_town,pays
0,Southampton,Angleterre
1,Cherbourg,France
2,Queenstown,Irlande


In [3]:
titanic_propre = pd.read_csv("../exercice_08/data/titanic_propre.csv")

In [7]:
nb_avant = len(titanic_propre)
print(nb_avant)

891


In [8]:
titanic_propre = titanic_propre.merge(
    ports,
    on="embark_town",
    how="left",
    validate="many_to_one"
)

In [10]:
assert len(titanic_propre) == nb_avant

In [11]:
resultat = titanic_propre.groupby("pays").agg(
    nombre_passagers=("survived", "size"),
    taux_survie=("survived", "mean")
)

resultat["taux_survie"] = (resultat["taux_survie"] * 100).round(2)

resultat

,nombre_passagers,taux_survie
pays,,
Angleterre,644,33.70
France,168,55.36
Irlande,77,38.96


In [14]:
ports_sans_queenstown = ports[
    ports["embark_town"] != "Queenstown"
]

ports_sans_queenstown
passagers = titanic_propre.drop(columns="pays")

titanic_inner = passagers.merge(
    ports_sans_queenstown,
    on="embark_town",
    how="inner"
)

print("Nombre de lignes :", len(titanic_inner))

Nombre de lignes : 812


In [16]:
titanic_left = passagers.merge(
    ports_sans_queenstown,
    on="embark_town",
    how="left"
)

print(titanic_left)

     survived  pclass     sex   age  sibsp  parch     fare embarked    who  \
0       False       3    male  22.0      1      0   7.2500        S    man   
1        True       1  female  38.0      1      0  71.2833        C  woman   
2        True       3  female  26.0      0      0   7.9250        S  woman   
3        True       1  female  35.0      1      0  53.1000        S  woman   
4       False       3    male  35.0      0      0   8.0500        S    man   
..        ...     ...     ...   ...    ...    ...      ...      ...    ...   
886     False       2    male  27.0      0      0  13.0000        S    man   
887      True       1  female  19.0      0      0  30.0000        S  woman   
888     False       3  female  21.5      1      2  23.4500        S  woman   
889      True       1    male  26.0      0      0  30.0000        C    man   
890     False       3    male  32.0      0      0   7.7500        Q    man   

     adult_male  embark_town  alone        pays  
0          Tr

### Comparaison entre les jointures inner et left

Après avoir retiré Queenstown du tableau des ports, j'ai effectué deux jointures.

- Avec `inner`, seuls les passagers dont le port existe dans le tableau `ports_sans_queenstown` sont conservés. Les passagers de Queenstown sont donc supprimés.
- Avec `left`, tous les passagers du tableau Titanic sont conservés. Cependant, les passagers de Queenstown ont une valeur manquante (`NaN`) dans la colonne `pays`.

**Conclusion :** la jointure `inner` supprime les lignes sans correspondance, tandis que la jointure `left` conserve toutes les lignes du tableau de gauche.